# Jiaozi Integration Update on Colab

This notebook deletes any existing `/content/Jiaozi` checkout, clones the latest `integration-recommender` branch, installs dependencies, and runs the integrated Jiaozi pipeline end to end on Colab.

The active flow is:

1. Module 1 parses the natural-language task with an LLM.
2. Module 2 analyzes the HuggingFace dataset.
3. Module 3 retrieves ranked CV model candidates.
4. Module 4 generates runnable training code.
5. The generated project runs the built-in smoke checks.


## Before running

1. In Colab, open **Secrets** and add `OPENAI_API_KEY`.
2. `KAGGLE_API_TOKEN` is optional here. The `integration-recommender` branch does not require Kaggle by default, but the notebook can store the token for future Kaggle workflows.
3. Select a GPU runtime if you want faster installs and later experimentation, though this notebook can complete on CPU.
4. Run the cells in order.


In [ ]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

def normalize_repo_url(value: str) -> str:
    value = (value or '').strip()
    markdown_match = re.fullmatch(r'\[(.*?)\]\((https?://[^)]+)\)', value)
    if markdown_match:
        return markdown_match.group(2)
    plain_url_match = re.search(r'https?://\S+', value)
    if plain_url_match:
        return plain_url_match.group(0)
    return value


REPO_URL = normalize_repo_url(os.getenv('JIAOZI_REPO_URL', 'https://github.com/Isso-W/Jiaozi.git'))
REPO_REF = os.getenv('JIAOZI_REPO_REF', 'integration-recommender')
REPO_DIR = Path('/content/Jiaozi')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
    check=True,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

print('Jiaozi repository:', REPO_DIR)
print('Jiaozi repo URL:', REPO_URL)
print('Jiaozi ref:', REPO_REF)


In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_secret(name: str, required: bool = False) -> str:
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            value = ''
    if not value and required:
        value = getpass(f'{name} (hidden input): ').strip()
    return value.strip()


openai_api_key = read_secret('OPENAI_API_KEY', required=True)
if not openai_api_key:
    raise RuntimeError('OPENAI_API_KEY is required for this notebook.')

os.environ['OPENAI_API_KEY'] = openai_api_key
os.environ['JIAOZI_LLM_PROVIDER'] = 'openai'
os.environ['M4_LLM_PROVIDER'] = 'openai'
os.environ.setdefault('M1_OPENAI_MODEL', 'gpt-4o')
os.environ.setdefault('M4_OPENAI_MODEL', 'gpt-5.5')

kaggle_token = read_secret('KAGGLE_API_TOKEN', required=False)
if kaggle_token:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token_path = kaggle_dir / 'access_token'
    token_path.write_text(kaggle_token, encoding='utf-8')
    token_path.chmod(0o600)
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    print('Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token')
else:
    print('No KAGGLE_API_TOKEN provided. This branch does not need Kaggle for the default run.')

del openai_api_key
del kaggle_token


## Persist the dataset on Google Drive

Mount Google Drive and point the HuggingFace cache (`HF_HOME`) at it. The dataset
is then downloaded **once** to Drive; later runs (even after the Colab runtime is
recycled) reuse it instead of re-downloading. `pipeline.py` and `run.py` run as
subprocesses and inherit `HF_HOME`, so both Module 2 analysis and training read
the same Drive cache.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Everything HuggingFace caches (datasets + hub) lives here, on Drive.
HF_CACHE_DIR = '/content/drive/MyDrive/Jiaozi/hf_cache'
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = os.path.join(HF_CACHE_DIR, 'datasets')

print('HF_HOME =', os.environ['HF_HOME'])
print('First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).')
print('Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.')

## Run the integrated pipeline

This sample run uses OpenAI for Module 1 parsing and `gpt-5.5` for Module 4 `model.py`
generation. The full chain runs end to end:

1. Module 1 parses `QUERY` into task type / priority / constraints.
2. Module 2 analyses `DATASET`.
3. Module 3 retrieves the ranked model candidates (backbone + head + loss + checkpoint).
4. Module 4 generates the runnable training project.

`QUERY` is the natural-language user input that drives Module 1 — edit it like a real
user would phrase the task. Do **not** hard-code a model name in it (e.g. "use mobilenet"),
otherwise Module 3 has nothing left to decide.

Set `REAL_TRAINING = True` to train the recommended model on the real dataset (GPU
recommended). With `REAL_TRAINING = False` the project is only generated and smoke-checked.

In [ ]:
import json

# ── User input (edit me) ───────────────────────────────────────────
# QUERY is the Module 1 input: describe the task in natural language, the way a real
# user would. Keep it neutral — don't name a backbone, let Module 3 choose.
QUERY = 'Classify images on a small dataset where accuracy matters more than speed.'
DATASET = 'uoft-cs/cifar10'

# Real training switch: True = run the full pipeline, then train the recommended
# model on the real dataset (GPU). False = only generate code + smoke check.
REAL_TRAINING = True
# Epochs for real training; None = use the value Module 3/pipeline recommended.
EPOCHS = None
# ───────────────────────────────────────────────────────────────────

OUTPUT_DIR = Path('/content/jiaozi_generated_pipeline')

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

command = [
    sys.executable,
    'pipeline.py',
    '--query', QUERY,
    '--dataset', DATASET,
    '--fmt', 'nl',
    '--module4-output', str(OUTPUT_DIR),
    '--module4-timeout', '120',
    '--module4-llm-provider', 'openai',
]
if REAL_TRAINING:
    command.append('--module4-real-training')

print('Running command:', ' '.join(command))
completed = subprocess.run(command, capture_output=True, text=True)
if completed.stdout:
    print('=== pipeline stdout ===')
    print(completed.stdout)
if completed.stderr:
    print('=== pipeline stderr ===')
    print(completed.stderr)
completed.check_returncode()

summary_path = OUTPUT_DIR / 'module4_summary.json'
if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    raise FileNotFoundError(
        f'Module 4 summary was not generated at {summary_path}. Available files: {available}'
    )

In [ ]:
summary_path = OUTPUT_DIR / 'module4_summary.json'
if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    print(f'Module 4 summary is missing: {summary_path}')
    print('Available files:', available)
else:
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2, ensure_ascii=False))

print('\nGenerated files:')
if OUTPUT_DIR.exists():
    for path in sorted(OUTPUT_DIR.iterdir()):
        print(path.name)
else:
    print('Output directory was not created.')


## Train the recommended model (real data)

This step only runs when `REAL_TRAINING = True`. It executes the generated `run.py`,
which loads the real dataset, trains the model Module 3 recommended, evaluates on the
test split, and saves checkpoints. Select a GPU runtime first for reasonable speed.

In [ ]:
if not REAL_TRAINING:
    print('REAL_TRAINING is False - skipping the real training step.')
    print('Set REAL_TRAINING = True in the run cell above to train on real data.')
else:
    import json

    # Persist checkpoints on Google Drive so they survive runtime recycling, and
    # resume automatically if a previous run left one. Set False to keep them in
    # the (ephemeral) generated project dir instead.
    SAVE_CHECKPOINTS_TO_DRIVE = True

    cfg_path = OUTPUT_DIR / 'configs.json'
    configs = json.loads(cfg_path.read_text(encoding='utf-8'))
    cfg = configs[0]
    mc = cfg.get('model_config', cfg)

    epochs = EPOCHS
    if epochs is None:
        epochs = mc.get('recommended_epochs') or cfg.get('recommended_epochs') or 10
    backbone = mc.get('backbone') or cfg.get('backbone')
    dataset_used = mc.get('dataset_id') or cfg.get('dataset_id') or DATASET

    if SAVE_CHECKPOINTS_TO_DRIVE and Path('/content/drive/MyDrive').exists():
        # Keyed by backbone + dataset so different runs don't clobber each other.
        tag = f"{backbone}_{str(dataset_used).replace('/', '_')}"
        ckpt_dir = f'/content/drive/MyDrive/Jiaozi/checkpoints/{tag}'
        os.makedirs(ckpt_dir, exist_ok=True)
        cfg['checkpoint_dir'] = ckpt_dir          # top-level -> read by train_model
        cfg['resume_checkpoint'] = 'auto'          # continue from last_checkpoint.pt if present
        cfg_path.write_text(json.dumps(configs, indent=2, ensure_ascii=False), encoding='utf-8')
        print('Checkpoints ->', ckpt_dir, '(resume: auto)')
    elif SAVE_CHECKPOINTS_TO_DRIVE:
        print('WARNING: Drive not mounted (run the Drive cell above); checkpoints will be ephemeral.')

    print(f'Training {backbone} for {epochs} epochs on {dataset_used} ...')
    train_command = [sys.executable, '-u', 'run.py', '--epochs', str(epochs)]  # -u: stream logs live
    print('Command:', ' '.join(train_command), '(cwd:', OUTPUT_DIR, ')\n')
    # No capture: stream epoch progress live into the notebook.
    subprocess.run(train_command, cwd=str(OUTPUT_DIR), text=True).check_returncode()
    print('\nReal training finished. Checkpoints under:', cfg.get('checkpoint_dir', 'checkpoints (ephemeral)'))

## Show the result

Reads the best checkpoint and prints its validation score **instantly** (the metric was
already computed during training — no re-evaluation). Set `FULL_EVAL = True` for a full
re-evaluation (macro-F1, params, sample count), which costs one extra pass over the eval set.

In [ ]:
import json, torch
from pathlib import Path

# Locate best_model.pt (prefer the Drive checkpoint dir set by the training cell).
_candidates = []
try:
    _candidates.append(Path(ckpt_dir) / 'best_model.pt')
except NameError:
    pass
_candidates.append(Path(OUTPUT_DIR) / 'checkpoints' / 'best_model.pt')
best_path = next((p for p in _candidates if p.exists()), None)
if best_path is None:
    raise FileNotFoundError(f'best_model.pt not found in: {[str(p) for p in _candidates]}')

ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
print('=== RESULT (best checkpoint) ===')
print('checkpoint :', best_path)
print('best_epoch :', ckpt.get('best_epoch'))
print('best_metric:', ckpt.get('best_metric'), '(validation metric at best epoch — no re-eval)')
print('validation :', json.dumps(ckpt.get('validation'), ensure_ascii=False))

# Full re-evaluation (macro_f1, params, num_samples) — one extra pass over the eval set.
FULL_EVAL = False
if FULL_EVAL:
    import os, sys
    os.chdir(OUTPUT_DIR); sys.path.insert(0, str(OUTPUT_DIR))
    from model import build_model
    from evaluate import evaluate
    _cfg = json.loads((Path(OUTPUT_DIR) / 'configs.json').read_text(encoding='utf-8'))[0]
    _cfg.update(_cfg.pop('model_config', {}) or {})
    _model = build_model(_cfg); _model.load_state_dict(ckpt['model_state_dict'])
    print('\n=== FULL EVALUATE ===')
    print(json.dumps(evaluate(_model, _cfg), indent=2, ensure_ascii=False))